# SHAP plots for the IRM nuisance models

Diagnostic figures for the interactive regression model, generated per recommendation.

- **Propensity model $m(X)$** (`propensity_*`): which baseline variables drive **who receives** the recommendation, i.e. confounding-by-indication. Produced for all 15 recommendations (the appendix set).
- **Outcome model $g_0(X)$** (`outcome_*`): which baseline variables **predict the outcome** among the untreated arm, i.e. prognosis. Produced only for the few worked confounding examples, paired with the propensity plot.

Neither is an interpretation of the causal effect: the forests are nuisance models and their feature importances carry no causal meaning. A variable that is prominent in **both** the propensity and the outcome plot is a confounder of the naive association. Outcome plots are deliberately not produced for every recommendation because, read alone, a prognostic plot is more misleading than informative.

Figures are written to `Outputs/shap/` as vector PDFs (scatter rasterized) plus high-resolution PNGs. Run the cells top to bottom; the first cell changes the working directory to the project root so the relative paths resolve.

In [1]:
import os
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

os.chdir("/Users/johannakastenberger/Library/Mobile Documents/com~apple~CloudDocs/Master Arbeit/master-thesis")

RNG = 123
np.random.seed(RNG)

DATA_PATH = "Outputs/dml_analysis_frame.parquet"
OUT_DIR   = "Outputs/shap"
os.makedirs(OUT_DIR, exist_ok=True)

## baseline confounders, identical to the DML script
W = [
    "anchor_age", "gender_male", "charlson_comorbidity_index",
    "sofa_score", "sapsii", "creatinine", "bun", "wbc", "platelets",
    "hemoglobin", "bilirubin", "albumin", "inr", "lactate",
    "ph_blood", "bicarbonate", "aki_baseline_stage",
]

## the 15 treatment indicators that enter the causal model as individual
## estimates (53 and 55 share one combined indicator)
treat_cols = [
    "rec_7_culture_before_antimicrobial_causal",
    "rec_8_lactate_measured_within_1h_causal",
    "rec_10_crystalloid_within_3h_causal",
    "rec_15_icu_6h_causal",
    "rec_16_antimicrobial_within_1h_causal",
    "rec_17_antimicrobial_within_1h_causal",
    "rec_44_balanced_over_saline_bolus_causal",
    "rec_53_55_norepi_first_line_causal",
    "rec_56_vasopressin_after_escalating_norepi_causal",
    "rec_68_first_oxygen_hfnc_causal",
    "rec_89_active_fluid_removal_7d_causal",
    "rec_90_restrictive_transfusion_causal",
    "rec_91_early_enteral_nutrition_causal",
    "rec_92_insulin_started_at_glucose_180_causal",
    "rec_96_sodium_bicarb_causal",
]

## readable titles, matching Table 7.1
titles = {
    "rec_7_culture_before_antimicrobial_causal":        "7: Culture before antimicrobial",
    "rec_8_lactate_measured_within_1h_causal":          "8: Lactate within 1 h",
    "rec_10_crystalloid_within_3h_causal":              "10: Crystalloid 30 mL/kg within 3 h",
    "rec_15_icu_6h_causal":                             "15: ICU admission within 6 h",
    "rec_16_antimicrobial_within_1h_causal":            "16: Antimicrobial within 1 h (shock)",
    "rec_17_antimicrobial_within_1h_causal":            "17: Antimicrobial within 1 h (sepsis)",
    "rec_44_balanced_over_saline_bolus_causal":         "44: Balanced over saline",
    "rec_53_55_norepi_first_line_causal":               "53/55: Norepinephrine first-line",
    "rec_56_vasopressin_after_escalating_norepi_causal": "56: Vasopressin added",
    "rec_68_first_oxygen_hfnc_causal":                  "68: HFNC over NIV initially",
    "rec_89_active_fluid_removal_7d_causal":             "89: Active fluid removal",
    "rec_90_restrictive_transfusion_causal":            "90: Restrictive transfusion",
    "rec_91_early_enteral_nutrition_causal":            "91: Early enteral nutrition",
    "rec_92_insulin_started_at_glucose_180_causal":      "92: Insulin at glucose >= 180",
    "rec_96_sodium_bicarb_causal":                      "96: Sodium bicarbonate",
}

def slug(treat_col):
    return treat_col.replace("rec_", "", 1).replace("_causal", "")

## readable feature labels (no underscores, abbreviations expanded,
## missingness flags spelled out)
pretty = {
    "anchor_age": "Age",
    "gender_male": "Male",
    "charlson_comorbidity_index": "Charlson comorbidity index",
    "sofa_score": "SOFA score",
    "sapsii": "SAPS II",
    "creatinine": "Creatinine",
    "bun": "BUN",
    "wbc": "WBC",
    "platelets": "Platelets",
    "hemoglobin": "Hemoglobin",
    "bilirubin": "Bilirubin",
    "albumin": "Albumin",
    "inr": "INR",
    "lactate": "Lactate",
    "ph_blood": "Blood pH",
    "bicarbonate": "Bicarbonate",
    "aki_baseline_stage": "AKI baseline stage",
}

def nice(name):
    if name.endswith("_miss"):
        base = name[:-5]
        return pretty.get(base, base.replace("_", " ")) + " (missing)"
    return pretty.get(name, name.replace("_", " "))

df = pd.read_parquet(DATA_PATH)
print(df.shape)

/opt/anaconda3/envs/thesis_dml/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(21189, 65)


In [2]:
def build_frame(treat_col, tier="full", outcome_col="mort_28d"):
    """Eligible stays for one recommendation, with the same tier filter,
    median imputation and missingness flags as run_dml in the DML script.
    Returns (d, X_cols)."""

    d = df.copy()
    d = d[d[treat_col].notna()]

    if tier == "well_observed":
        d = d[d["well_observed"] == True]
    elif tier == "well_observed_strict":
        d = d[d["well_observed_strict"] == True]

    d = d.rename(columns={treat_col: "A", outcome_col: "Y"})
    d = d[d["A"].notna()]
    d["A"] = d["A"].astype(int)

    X_cols = []
    for col in W:
        miss = d[col].isna()
        if miss.any():
            flag = col + "_miss"
            d[flag] = miss.astype(int)
            d.loc[miss, col] = d[col].median()
            X_cols.append(flag)
        X_cols.append(col)

    return d, X_cols


def _style_and_save(explanation, title, out_base, show=True):
    """Shared styling: readable labels, small rasterized dots, vector PDF.
    Pass show=False during batch runs to skip inline rendering (faster)."""

    explanation.feature_names = [nice(n) for n in explanation.feature_names]

    shap.plots.beeswarm(explanation, max_display=12, show=False)

    fig = plt.gcf()
    ax  = plt.gca()
    fig.set_size_inches(7, 5)

    for coll in ax.collections:
        coll.set_sizes([1.5])
        coll.set_rasterized(True)

    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_base + ".pdf", dpi=400, bbox_inches="tight")
    plt.savefig(out_base + ".png", dpi=400, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)

In [3]:
import time

## DIAGNOSTIC forest, deliberately smaller than the inference nuisance so
## TreeSHAP is fast. Fewer, shallower trees (large leaves) change which
## variables top the plot only marginally. Raise toward n_estimators=500,
## min_samples_leaf=5 if you want it to mirror the IRM nuisance exactly.
N_TREES  = 150
MIN_LEAF = 50

## cap on how many rows the explainer runs over. Set to None to explain all.
MAX_EXPLAIN = 2000

def _explain_rows(X):
    if MAX_EXPLAIN is not None and len(X) > MAX_EXPLAIN:
        return X.sample(n=MAX_EXPLAIN, random_state=RNG)
    return X


def propensity_shap(treat_col, tier="full", show=True, verbose=True):
    """SHAP for the treatment-assignment model m(X): who gets the recommendation."""

    d, X_cols = build_frame(treat_col, tier)
    X, A = d[X_cols], d["A"]

    t0 = time.time()
    m = RandomForestClassifier(n_estimators=N_TREES, min_samples_leaf=MIN_LEAF,
                               random_state=RNG, n_jobs=-1)
    m.fit(X, A)
    t1 = time.time()

    Xs = _explain_rows(X)
    expl = shap.TreeExplainer(m, feature_perturbation="tree_path_dependent")
    sv = expl(Xs, check_additivity=False)[..., 1]   # P(treated)
    t2 = time.time()

    if verbose:
        print(f"{slug(treat_col):40s} fit {t1-t0:4.1f}s  shap {t2-t1:4.1f}s  "
              f"(n={len(X)}, explained={len(Xs)})")

    title = "Treatment assignment, " + titles.get(treat_col, treat_col)
    _style_and_save(sv, title, f"{OUT_DIR}/propensity_{slug(treat_col)}", show=show)


def outcome_shap(treat_col, tier="full", outcome_col="mort_28d", show=True, verbose=True):
    """SHAP for the outcome nuisance g0(X): prognosis among the UNTREATED arm."""

    d, X_cols = build_frame(treat_col, tier, outcome_col)
    ctrl = d[(d["A"] == 0) & d["Y"].notna()]
    Xg, Yg = ctrl[X_cols], ctrl["Y"].astype(float)

    t0 = time.time()
    g0 = RandomForestRegressor(n_estimators=N_TREES, min_samples_leaf=MIN_LEAF,
                               random_state=RNG, n_jobs=-1)
    g0.fit(Xg, Yg)
    t1 = time.time()

    Xs = _explain_rows(Xg)
    expl = shap.TreeExplainer(g0, feature_perturbation="tree_path_dependent")
    sv = expl(Xs, check_additivity=False)
    t2 = time.time()

    if verbose:
        print(f"{slug(treat_col):40s} fit {t1-t0:4.1f}s  shap {t2-t1:4.1f}s  "
              f"(ctrl n={len(Xg)}, explained={len(Xs)})")

    title = "Outcome (untreated arm), " + titles.get(treat_col, treat_col)
    _style_and_save(sv, title, f"{OUT_DIR}/outcome_{slug(treat_col)}", show=show)

In [9]:
from tqdm.auto import tqdm

## 1) propensity SHAP for all 15 recommendations -> appendix set.
##    show=False: save only, no inline render (much faster for a batch)
for tc in tqdm(treat_cols, desc="propensity (all recs)"):
    propensity_shap(tc, show=False)

## 2) propensity + outcome pairs for the worked confounding examples:
##    53/55 and 90 are explained by severity (confounded),
##    44 is a bundle effect that should NOT be severity-driven (clean contrast)
pair_recs = [
    "rec_53_55_norepi_first_line_causal",
    "rec_90_restrictive_transfusion_causal",
    "rec_44_balanced_over_saline_bolus_causal",
]
for tc in tqdm(pair_recs, desc="outcome (pairs)"):
    outcome_shap(tc, show=False)

print("done -> Outputs/shap/")

propensity (all recs):   0%|          | 0/15 [00:00<?, ?it/s]

7_culture_before_antimicrobial           fit  0.3s  shap  4.7s  (n=18038, explained=2000)


propensity (all recs):   7%|▋         | 1/15 [00:05<01:17,  5.55s/it]

8_lactate_measured_within_1h             fit  0.3s  shap  6.1s  (n=21189, explained=2000)


propensity (all recs):  13%|█▎        | 2/15 [00:12<01:22,  6.36s/it]

...

90_restrictive_transfusion               fit  0.2s  shap  0.8s  (ctrl n=4702, explained=2000)


outcome (pairs):  67%|██████▋   | 2/3 [00:02<00:01,  1.40s/it]

44_balanced_over_saline_bolus            fit  0.1s  shap  0.1s  (ctrl n=1120, explained=1120)


outcome (pairs): 100%|██████████| 3/3 [00:03<00:00,  1.10s/it]

done -> Outputs/shap/
